# Pose 키포인트 기반 팔굽혀펴기 횟수 카운터

이 노트북은 **직접 촬영한 스마트폰 팔굽혀펴기 영상**을 대상으로, YOLO Pose로 관절 키포인트를 추출하고 **팔꿈치 각도**를 이용해 횟수를 세는 실습용 노트북입니다.

핵심 흐름은 다음과 같습니다.

1. 스마트폰 영상을 전처리해 해상도와 FPS를 줄입니다.
2. YOLO Pose로 프레임별 관절 좌표를 추출합니다.
3. `어깨-팔꿈치-손목` 각도를 계산해 1차원 시계열로 바꿉니다.
4. 이동평균 스무딩과 임계값을 조정해 팔굽혀펴기 횟수를 셉니다.
5. 정답 횟수를 직접 입력했다면 오차도 함께 비교합니다.


## 0. 환경 설정

- 스마트폰 영상은 해상도가 커서 바로 돌리면 느릴 수 있으므로 전처리 셀을 먼저 제공했습니다.
- 팔굽혀펴기는 **측면 촬영**이 가장 좋습니다.
- 오른팔이 잘 보인다면 기본 관절 조합 `(6, 8, 10)`을 그대로 사용하면 됩니다.
- 실제 횟수를 알고 있다면 `GROUND_TRUTH_REPS`에 직접 넣어두면 됩니다.


In [ ]:
# 필요 시 1회만 실행
# %pip install -q ultralytics opencv-python matplotlib numpy pandas


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 본인 스마트폰 영상 파일명으로 바꾸세요.
VIDEO_PATH = "my_pushup_video.mp4"

# 팔굽혀펴기 기준: 오른어깨(6)-오른팔꿈치(8)-오른손목(10)
JOINT_TRIPLE = (6, 8, 10)

# 직접 세어본 정답 횟수. 아직 모르면 None으로 두세요.
GROUND_TRUTH_REPS = None

# 시작용 기본값. 영상마다 조정해야 합니다.
DOWN_THR = 95
UP_THR = 150

model = YOLO("yolov8n-pose.pt")
print("모델 로드 완료")


## 1. 스마트폰 영상 전처리

스마트폰 영상은 보통 해상도와 FPS가 커서 포즈 추출 속도가 느립니다. 아래 셀은 OpenCV만 이용해서:

- 긴 변 기준 해상도 축소
- FPS 축소
- 세로 촬영 영상 회전
- 너무 긴 영상 프레임 제한

을 수행한 뒤 새 mp4를 저장합니다.


In [ ]:
USE_PREPROCESSED_VIDEO = True
INPUT_VIDEO_PATH = VIDEO_PATH
OUTPUT_VIDEO_PATH = "processed_" + INPUT_VIDEO_PATH.rsplit('.', 1)[0] + ".mp4"

PREPROCESS_CONFIG = {
    "max_side": 960,
    "target_fps": 15,
    "rotate_code": None,
    "max_frames": None,
}

def preprocess_video_for_pose(input_path, output_path, max_side=960, target_fps=15, rotate_code=None, max_frames=None):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"비디오를 열 수 없습니다: {input_path}")

    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    src_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    src_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    scale = min(1.0, max_side / max(src_w, src_h))
    out_w = max(2, int(round(src_w * scale)))
    out_h = max(2, int(round(src_h * scale)))
    out_w += out_w % 2
    out_h += out_h % 2

    if rotate_code in (cv2.ROTATE_90_CLOCKWISE, cv2.ROTATE_90_COUNTERCLOCKWISE):
        writer_size = (out_h, out_w)
    else:
        writer_size = (out_w, out_h)

    target_fps = min(float(target_fps), float(src_fps)) if src_fps > 0 else float(target_fps)
    frame_step = max(1, int(round(src_fps / target_fps))) if target_fps > 0 else 1
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, max(target_fps, 1.0), writer_size)

    frame_idx = 0
    saved = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_step != 0:
            frame_idx += 1
            continue
        if scale != 1.0:
            frame = cv2.resize(frame, (out_w, out_h), interpolation=cv2.INTER_AREA)
        if rotate_code is not None:
            frame = cv2.rotate(frame, rotate_code)
        writer.write(frame)
        saved += 1
        frame_idx += 1
        if max_frames is not None and saved >= max_frames:
            break

    cap.release()
    writer.release()
    return {
        "input_path": input_path,
        "output_path": output_path,
        "src_size": (src_w, src_h),
        "dst_size": writer_size,
        "src_fps": src_fps,
        "dst_fps": max(target_fps, 1.0),
        "saved_frames": saved,
    }

ANALYSIS_VIDEO_PATH = VIDEO_PATH
if USE_PREPROCESSED_VIDEO:
    info = preprocess_video_for_pose(INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH, **PREPROCESS_CONFIG)
    ANALYSIS_VIDEO_PATH = info["output_path"]
    print("전처리 완료")
    print("입력:", info["input_path"], "| 출력:", info["output_path"])
    print("해상도:", info["src_size"], "->", info["dst_size"])
    print("FPS:", round(info["src_fps"], 2), "->", round(info["dst_fps"], 2))
    print("저장 프레임 수:", info["saved_frames"])
else:
    print("전처리 생략")
    print("분석 영상:", ANALYSIS_VIDEO_PATH)


## 2. 키포인트를 각도로 바꾸기

팔굽혀펴기 카운팅에서는 보통 **팔꿈치가 얼마나 굽혀졌는가**가 중요하므로 `어깨-팔꿈치-손목` 각도를 사용합니다.

- 위로 올라온 상태: 각도가 큼
- 아래로 내려간 상태: 각도가 작음


In [ ]:
def calc_angle(a, b, c):
    """b를 꼭짓점으로 하는 세 점의 각도(도)를 반환"""
    a, b, c = np.array(a, float), np.array(b, float), np.array(c, float)
    ba, bc = a - b, c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))

def extract_angle_series(video_path, joint_triple):
    a_idx, b_idx, c_idx = joint_triple
    cap = cv2.VideoCapture(video_path)
    angles = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        res = model(frame, verbose=False)
        kp = res[0].keypoints.xy.cpu().numpy()
        if len(kp) > 0 and kp[0].shape[0] == 17:
            p = kp[0]
            a, b, c = p[a_idx], p[b_idx], p[c_idx]
            if a.sum() > 0 and b.sum() > 0 and c.sum() > 0:
                angles.append(calc_angle(a, b, c))
            else:
                angles.append(angles[-1] if angles else 180.0)
        else:
            angles.append(angles[-1] if angles else 180.0)
    cap.release()
    return np.array(angles)


## 3. 원시 각도 신호 확인

먼저 포즈 추출이 잘 되었는지 전체 각도 범위와 파형을 확인합니다. 이 그래프를 보고 대략적인 `DOWN_THR`, `UP_THR`를 정하게 됩니다.


In [ ]:
raw_angles = extract_angle_series(ANALYSIS_VIDEO_PATH, JOINT_TRIPLE)
print("프레임 수:", len(raw_angles))
print("각도 범위:", round(raw_angles.min(), 1), "~", round(raw_angles.max(), 1), "도")

plt.figure(figsize=(12, 4))
plt.plot(raw_angles, linewidth=1)
plt.title("Raw elbow angle over frames")
plt.xlabel("frame")
plt.ylabel("angle (deg)")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(12, 3))
zoom_seg = raw_angles[:min(150, len(raw_angles))]
plt.plot(zoom_seg, marker='.', markersize=3)
plt.title("Zoom-in: frame-to-frame jitter")
plt.xlabel("frame")
plt.ylabel("angle (deg)")
plt.grid(alpha=0.3)
plt.show()


## 4. 스무딩

팔꿈치 각도에도 프레임 단위 노이즈가 들어갈 수 있으므로 이동평균으로 신호를 부드럽게 만듭니다. 다만 윈도우가 너무 크면 실제 팔굽혀펴기 움직임까지 뭉개질 수 있습니다.


In [ ]:
def moving_average(x, window):
    if window <= 1:
        return x.copy()
    kernel = np.ones(window) / window
    padded = np.pad(x, (window // 2, window // 2), mode='edge')
    smoothed = np.convolve(padded, kernel, mode='same')
    return smoothed[window // 2 : window // 2 + len(x)]

plt.figure(figsize=(12, 4))
plt.plot(raw_angles, alpha=0.5, label="raw")
plt.plot(moving_average(raw_angles, 5), linewidth=2, label="smoothed (w=5)")
plt.title("Raw vs Smoothed elbow angle")
plt.xlabel("frame")
plt.ylabel("angle (deg)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 5. 상태머신 기반 팔굽혀펴기 카운터

팔이 충분히 굽혀지면 `down`, 다시 충분히 펴지면 `up`으로 상태를 바꾸면서 `up` 복귀 시 1회를 증가시킵니다.


In [ ]:
def count_reps(angle_series, down_thr, up_thr):
    stage = "up"
    count = 0
    for ang in angle_series:
        if ang < down_thr and stage == "up":
            stage = "down"
        if ang > up_thr and stage == "down":
            stage = "up"
            count += 1
    return count

raw_count = count_reps(raw_angles, DOWN_THR, UP_THR)
print(f"[raw 신호] 카운트 = {raw_count}")
if GROUND_TRUTH_REPS is not None:
    print(f"정답 = {GROUND_TRUTH_REPS}, 절대 오차 = {abs(raw_count - GROUND_TRUTH_REPS)}")
else:
    print("GROUND_TRUTH_REPS가 None이라 오차 계산은 생략합니다.")


## 6. 스무딩 윈도우별 비교

직접 촬영한 영상에서도 스무딩이 도움이 되는지 확인합니다. 정답 횟수를 입력했다면 오차까지 같이 비교됩니다.


In [ ]:
def abs_error(pred, truth):
    return None if truth is None else abs(pred - truth)

windows = [1, 3, 5, 7, 9, 11, 15, 21]
results = []
for w in windows:
    sm = moving_average(raw_angles, w)
    cnt = count_reps(sm, DOWN_THR, UP_THR)
    err = abs_error(cnt, GROUND_TRUTH_REPS)
    results.append((w, cnt, err))

print(f"{'window':>7} | {'predicted':>9} | {'abs_error':>9}")
print('-' * 33)
for w, cnt, err in results:
    err_text = err if err is not None else '-'
    print(f"{w:>7} | {cnt:>9} | {str(err_text):>9}")

ws = [r[0] for r in results]
cnts = [r[1] for r in results]
errs = [np.nan if r[2] is None else r[2] for r in results]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(ws, cnts, marker='o')
ax[0].set_title('Predicted push-up reps vs smoothing window')
ax[0].set_xlabel('window size')
ax[0].set_ylabel('reps')
ax[0].grid(alpha=0.3)

ax[1].plot(ws, errs, marker='s', color='darkorange')
ax[1].set_title('Absolute error vs smoothing window')
ax[1].set_xlabel('window size')
ax[1].set_ylabel('|error|')
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

if GROUND_TRUTH_REPS is not None:
    valid_results = [r for r in results if r[2] is not None]
    best = min(valid_results, key=lambda r: r[2])
    print(f"가장 오차가 작은 윈도우: w={best[0]} (예측 {best[1]}, 오차 {best[2]})")
else:
    best = max(results, key=lambda r: r[1])
    print("정답이 없어 오차 기반 best window는 계산하지 않았습니다.")
    print(f"참고용 최대 예측 카운트 윈도우: w={best[0]} (예측 {best[1]})")


## 7. 임계값 민감도 실험

팔굽혀펴기 영상마다 자세와 카메라 위치가 달라서 threshold가 크게 달라질 수 있습니다. 아래 셀은 `down threshold`를 바꾸며 얼마나 민감한지 확인합니다.


In [ ]:
best_w = best[0]
best_smoothed = moving_average(raw_angles, best_w)

down_candidates = [70, 80, 90, 100, 110, 120]
print(f"(window={best_w} 고정, up_thr={UP_THR} 고정)")
print(f"{'down_thr':>8} | {'predicted':>9} | {'abs_error':>9}")
print('-' * 34)

th_errs = []
for d in down_candidates:
    cnt = count_reps(best_smoothed, d, UP_THR)
    err = abs_error(cnt, GROUND_TRUTH_REPS)
    th_errs.append(np.nan if err is None else err)
    err_text = err if err is not None else '-'
    print(f"{d:>8} | {cnt:>9} | {str(err_text):>9}")

plt.figure(figsize=(7, 4))
plt.plot(down_candidates, th_errs, marker='o')
plt.title('Absolute error vs down-threshold')
plt.xlabel('down threshold (deg)')
plt.ylabel('|error|')
plt.grid(alpha=0.3)
plt.show()


## 8. 제출용 또는 기록용 정리

아래 문장을 실행 결과에 맞게 채우면 됩니다.

1. 원시 팔꿈치 각도 신호에서 프레임 간 떨림이 __보였다 / 거의 없었다__.
2. 기본 threshold(`DOWN_THR`, `UP_THR`)에서 raw 카운트는 `__회`였다.
3. 스무딩 window `__`에서 가장 안정적인 결과가 나왔다.
4. `down threshold`는 `__도` 부근에서 가장 적절했다.
5. 한계점: 직접 촬영 영상은 촬영 각도, 가림, 조명, 속도 차이에 따라 결과가 달라질 수 있다.

권장 실험 순서:

- 먼저 그래프를 보고 `DOWN_THR`, `UP_THR`를 대략 정합니다.
- raw 카운트가 틀리면 threshold를 먼저 조정합니다.
- 그다음 스무딩 윈도우를 조절합니다.
- 마지막으로 정답 횟수를 입력해 오차를 비교합니다.
